Létrehozta: Panna

Adatok beolvasása:

In [ ]:
from pathlib import Path
from services.data_loader import load_messages_by_year, merge_years

PROJECT_ROOT = Path(globals().get("project_root", ".")).resolve()
JSON_DIR = PROJECT_ROOT / "data" / "json"

yearly, names = load_messages_by_year(JSON_DIR)
merged = merge_years(yearly)

1. Mi volt Anita utolsó üzenete? (1 pont)

In [16]:
uzik = [i for i in merged if i['sender_name'] == "Anita Kaderják"]

uzik.sort(key=lambda x: x['timestamp_ms'])

for i in reversed(uzik[-5:]):
    print(i['content'])

sajnos idén már olyat sem talált, aki felvezette volna, de azért kapott 3 szavazatot

Mi kaptunk egy ilyet: Tisztelt Címzett!

A Nemzeti Kiberbiztonsági Intézet tájékoztatása alapján a PHP-t futtató környezetben kritikus hibát fedeztek fel, amely globálisan érinti a rendszereket. A rendszerek védelme érdekében az Ön felügyelete alá tartozó honlap/ok leállításra kerültek. A hiba elhárítása érdekében a rendszerelem frissítése folyamatban van. A sérülékenység megszüntetését követően tájékoztatást küldünk a rendszer újraindulásáról.
Van tűz?
hejho! 9től a pinceben leszek a szívügyem és a kocsma társaságában


2. Kiket távolított el eddig Attila a csoportból? (2 pont)

In [11]:
kickelt = []
for uzenet in merged:
    if uzenet["content"] != 0:
        szavak = uzenet["content"].split()
        if "eltávolította" in szavak and uzenet["sender_name"] == "Attila Chikan":
            kickelt.append(uzenet)

for egyedi in kickelt:
    print(egyedi["content"].split(":")[1].strip("."))

 János Biró
 Zoli Laufer
 Kégl Virág
 Gáspár Harmati
 Ági Tajti


3. Mi a legtöbbet használt reakció? (2 pont)

In [80]:
jelek = {}

for i in merged:
    for j in i['reactions']:
        jel = j['reaction']
        jelek[jel] = jelek.get(jel, 0) + 1

sorted_jelek = sorted(jelek.items(), key=lambda x: x[1], reverse=True)

for reaction, count in sorted_jelek[:5]:
    print(reaction)

❤
😆
😍
👍
😢


4. Összesen melyik évfolyam mennyi üzenetet küldött? (4 pont)

In [85]:
from collections import Counter

nev_evfolyam = {n['name']: n['class'] for n in names}

evfolyamok = Counter()

for i in merged:
    if len(i.get('content', '')) > 0:
        nev = i['sender_name']
        if nev in nev_evfolyam:
            evfolyamok[nev_evfolyam[nev]] += 1

# Rendezés évfolyamszám szerint csökkenően
for evfolyam, db in sorted(evfolyamok.items(), reverse=True)[0:5]:
    print(f"{db}")

19
293
1388
2957
5554


5. Ki írta a legtöbb üzenetet? (2 pont)

In [78]:
from collections import Counter

uzenetek = Counter(
    i['sender_name']
    for i in merged
    if len(i.get('content', '')) > 0
)

rendezett = uzenetek.most_common()

if rendezett:
    hatar = rendezett[min(4, len(rendezett) - 1)][1]

    for nev, db in rendezett:
        if db >= hatar:
            print(nev)

Ákos Ozoróczy
Eszter Kaló
Gergely Magyar
Veronika Ipacs
Barnabás Muzslai


6. Ki vonta vissza a legtöbb üzenetet? (1 pont)

In [65]:
visszavonok = {}

for i in merged:
    if i['is_unsent'] > 0:
        name = i['sender_name']
        visszavonok[name] = visszavonok.get(name, 0) + 1

sorted_visszavonok = sorted(
    visszavonok.items(),
    key=lambda x: x[1],
    reverse=True
)

for name, count in sorted_visszavonok[:5]:
    print(name)

Herbák Mihály
Barnabás Muzslai
Csaba Jázmin
Főző Zsolt
Julian Maxim Zell


7. Mennyi a legtöbb idő ami eltelt két egymást követő üzenet között? (4 pont)

In [64]:
timestamps = sorted(i['timestamp_ms'] for i in merged)

szunetek = [
    timestamps[i+1] - timestamps[i]
    for i in range(len(timestamps) - 1)
]

top5 = sorted(szunetek, reverse=True)[:5]

for x in top5:
    print(x)

733456577
485197076
423253096
405675042
370900396


8. Ki írja átlagosan a leghosszabb üzeneteket? Itt nem számítanak a 0 karakter hosszú üzenetek. (2 pont)

In [63]:
hosszu = {}

for i in merged:
    name = i['sender_name']
    content = i.get('content', "")

    if content:
        hosszu.setdefault(name, []).append(len(content))

atlagok = {
    name: sum(lengths) / len(lengths)
    for name, lengths in hosszu.items()
    if lengths
}

top5 = sorted(atlagok.items(), key=lambda x: x[1], reverse=True)[:5]

for name, avg in top5:
    print(name)

Jázmin Miczki
Barnabás Felicián Tóth
Adri Czeilinger
Bence B Rottek
Péter Topercer


9. Ki lépett ki a legtöbbször a jeszk momentsből (2 pont)

In [62]:
ki = {}

for i in merged:
    content = i.get('content')

    if content and 'kilépett' in content:
        name = i['sender_name']
        ki[name] = ki.get(name, 0) + 1

top5 = sorted(
    ki.items(),
    key=lambda x: (-x[1], x[0])
)[:5]

for name, count in top5:
    print(name)

Bátor Adri
Pálfi Lóránt
Roxána Lévai
Réka Nagy
Anna Maria Džunić


10. Ki küldte legtöbb képet a jeszk momentsbe? (2 pont)

In [61]:
kepkuldok = {}

for i in merged:
    photos = i.get('photos', 0)

    if photos > 0:
        name = i['sender_name']
        kepkuldok[name] = kepkuldok.get(name, 0) + photos

top5 = sorted(kepkuldok.items(), key=lambda x: (-x[1], x[0]))[:5]

for name, count in top5:
    print(name)

Barnabás Muzslai
Hoffmann Kornél
Eszter Kaló
Attila Chikan
Zsuzsanna Orehóczki


11. Hányan vannak, akik pontosan n évben küldtek üzenetet ($ n = 1, ... 7 $) (2 pont, ábrázolásért +1) (2 pont)

In [60]:
years = [
    yearly["2017"],
    yearly["2018"],
    yearly["2019"],
    yearly["2020"],
    yearly["2021"],
    yearly["2022"],
    yearly["2023"],
    yearly["2024"],
    yearly["2025"],
    yearly["2026"]
]

user_years = {}

for year_index, data in enumerate(years, start=2017):
    for msg in data:
        name = msg['sender_name']

        if name not in user_years:
            user_years[name] = set()

        user_years[name].add(year_index)

year_counts = [len(s) for s in user_years.values()]

result = {i: 0 for i in range(1, 11)}

for c in year_counts:
    if 1 <= c <= 10:
        result[c] += 1

for i in range(1, 11):
    print(i, result[i])

1 31
2 40
3 37
4 49
5 32
6 41
7 42
8 15
9 9
10 5


12. Melyik szó volt a legnépszerűbb az egyes években? (4 pont)

In [59]:
years = {
    2017: yearly["2017"],
    2018: yearly["2018"],
    2019: yearly["2019"],
    2020: yearly["2020"],
    2021: yearly["2021"],
    2022: yearly["2022"],
    2023: yearly["2023"],
    2024: yearly["2024"],
    2025: yearly["2025"],
    2026: yearly["2026"]
}

for year, data in years.items():
    freq = {}

    for msg in data:
        text = msg.get('content_clean', "")

        if text:
            for word in text.lower().split():
                freq[word] = freq.get(word, 0) + 1

    top5 = sorted(freq.items(), key=lambda x: (-x[1], x[0]))[:1]
    
    for word, count in top5:
        print(word)

pac-man
💚
💚
💚
gyertek
gyertek
lehet
lehet
lehet
lehet


13. Mi volt az az üzenet, amelyik a legtöbb reakciót kapta? (2 pont)

In [58]:
reakciok = []

for i in merged:
    szam = len(i.get('reactions', []))
    reakciok.append(szam)

reakciok.sort()
top5_vals = reakciok[-5:][::-1]  # biggest → smallest

# collect matching posts
top_dicts = {v: [] for v in top5_vals}

for i in merged:
    szam = len(i.get('reactions', []))
    if szam in top_dicts:
        top_dicts[szam].append(i)

# print results
for v in top5_vals:
    #print("\nREACTIONS =", v)
    print(top_dicts[v][0]["content"])

Arijovwode Ervin Ovie, Chikán Eszter és Obovi Arijovwode második gyermeke, nekünk negyedik unokánk. 3950/57, minden rendben🙂
Boldog karácsonyt kívánok a Rajk népének!
Remélem, midenki szerencsésen hazaért, s nem hagyott semmi fontosat a táborban. A rövid távú visszatekitében én nagyon jónak tartom a tábort. Persze lehetnek kifogások (a viták egyenlőtlen színvonalától kezdve a kajáig), de különösen a zárónap (az évértékelés, a diplomaátadó és a tábortűz) fényében a magam részéről öszességében örömmel tekintek vissza erre a pár napra. Köszönet mindenkinek, aki ezt lehetővé tette.
Jó hír: ma délelőtt megkaptuk a Pfizer első adagját  Mártával. Három hét múlva jön a második, aztán még két hetet kell várni, s mehetek a kommunába🙂



14. Meddig tartott a leghosszabb hívás? (2 pont)

In [57]:
timestamps = []

for i in merged:
    content = i.get('content', "")

    if content:
        is_start = (
            ("hívást indított." in content) or
            ("videochatet indított." in content)
        )

        is_end = (
            ("hívás befejeződött." in content) or
            ("videohívás véget ért." in content) or
            ("videóhívás véget ért." in content)
        )

        if is_start or is_end:
            timestamps.append(i['timestamp_ms'])

timestamps.sort()

hivas_hossza = [
    timestamps[i+1] - timestamps[i]
    for i in range(0, len(timestamps) - 1, 2)
]

hivas_hossza.sort()

print(hivas_hossza[-1] // 1000)
print(hivas_hossza[-2] // 1000)
print(hivas_hossza[-3] // 1000)
print(hivas_hossza[-4] // 1000)
print(hivas_hossza[-5] // 1000)

16835861
4301557
2937809
511886
373799


15. Ki érte el a legtöbb reakciót átlagosan a jeszk momentsben? (4 pont)

In [55]:
nepszeruek = {}

for i in merged:
    name = i['sender_name']
    szam = len(i.get('reactions', []))

    if name not in nepszeruek:
        nepszeruek[name] = []

    nepszeruek[name].append(szam)

# compute averages
atlagok = {
    name: sum(values) / len(values)
    for name, values in nepszeruek.items()
    if values
}

top5 = sorted(atlagok.items(), key=lambda x: (-x[1], x[0]))[:5]

for name, avg in top5:
    print(name)

Drong Lilla
Réka Asztalos
Mate Meszaros
Marton Kovesdy
Anita Kaderják


16. Melyik nap küldték a legtöbb üzenetet a jeszk momentsbe? (1 pont)

In [54]:
osszesites = {}

for i in merged:
    datum = f"{i['year']}.{i['month']}.{i['day']}"
    osszesites[datum] = osszesites.get(datum, 0) + 1

top5 = sorted(osszesites.items(), key=lambda x: (-x[1], x[0]))[:5]

for rank, (datum, count) in enumerate(top5, start=1):
    print(datum)

2017.10.17
2019.2.6
2020.3.18
2026.7.4
2025.7.1


17. Mennyi üzenetet küldenek átlagosan egy nap a jeszk momentsbe? (1 pont)

In [53]:
osszesites = {}

for i in merged:
    datum = (i['year'], i['month'], i['day'])

    osszesites[datum] = osszesites.get(datum, 0) + 1

total_messages = len(merged)
unique_days = len(osszesites)

avg = total_messages / unique_days

print(avg)
#print(total_messages)
#print(unique_days)

23.616007474307068


18. Hányszor végzett valamilyen aktivitást 2026-ben egy jelenlegi DB vagy SZMT tag (az igazgatókat nem beleértve) (1 pont)

In [52]:
osszes = 0

for i in merged:
    nev = i['sender_name']

    if nev in [
        "Rőfös-Horvát Judit",
        "Palkó Pauer",
        "Tamás Végh",
        "Dorka Pataki",
        "Máté Fischer",
        "Kincső Boróka Pénzes",
        "Noa Cernov",
        "Liza Almási",
        "Bernadett Belényesi",
        "Novák Levente",
        "Lili Probojcsevity",
        "Bálint Fazekas",
        "Virág Imre János",
        "Zoltán Deskó",
        "Flavia Vassallo",
        "Solymi Vigh"
    ]:
        if i['year'] == 2026:
            osszes += 1

print(osszes)

2436


19. Ki káromkodott a legtöbbet az összes beküldött üzenetével arányosan?  swear_words = ["szar","fasz","basz","geci", "kurva","fos"] (1 pont)

In [51]:
swear_words = {"szar", "fasz", "basz", "geci", "kurva", "fos"}

csunya = {}
ossz_uz = {}

for i in merged:
    name = i['sender_name']
    content = i.get('content', "")

    ossz_uz[name] = ossz_uz.get(name, 0) + 1

    if content:
        words = content.split()

        if any(w in words for w in swear_words):
            csunya[name] = csunya.get(name, 0) + 1

# ratio per user
ratios = {
    name: csunya.get(name, 0) / ossz_uz[name]
    for name in ossz_uz
}

# top 5 (ratio desc, name asc tie-break)
top5 = sorted(
    ratios.items(),
    key=lambda x: (-x[1], x[0])
)[:5]

for name, ratio in top5:
    print(name)

Zsófi Szalai
Szeibert Alexandra
Norbert Tollner
Sandor Lukacs
Bálint Fazekas


20. Milyen arányban reagálnak lányok, lányok és fiúk üzeneteire, és vice-versa? (Szedd össze az összes üzenetet lányoktól és milyen arányban reagálnak rájuk lányok és fiúk és ezt csináld meg a faszik estében is) (4 pont)

In [50]:
gender_map = {p['name']: p['gender'] for p in names}

counts = {'f': 0, 'n': 0}

for msg in merged:
    sender = msg['sender_name']

    # only girls' messages
    if gender_map.get(sender) == 'n':

        for reaction in msg.get('reactions', []):
            actor = reaction['actor']
            gender = gender_map.get(actor)

            if gender in counts:
                counts[gender] += 1

#print(counts)

total = counts['f'] + counts['n']

if total > 0:
    ratio = counts['f'] / total
else:
    ratio = 0

print(ratio)

0.4918295712989864


Függvényes feladatok

21. Megadok egy timestamp-et, melyik az eddig az időpontig leggyakrabban használt szó? (2 pont)

In [49]:
def get_most_common_words_so_far(timestamp_ms):
    word_counts = {}

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            text = message.get('content_clean', "")

            for word in text.split():
                word_counts[word] = word_counts.get(word, 0) + 1

    top5 = sorted(
        word_counts.items(),
        key=lambda x: (-x[1], x[0])
    )[:5]

    return top5

tuples = get_most_common_words_so_far(1726853539100)

for t in tuples:
    print(t[0])

💚
lehet
gyertek
nagyon
én


22. Megadok egy timestamp-et, melyik évfolyam volt eddig az időpontig a legaktívabb, a legalább egy üzenetet küldő emberekre átlagolva? (4 pont)

In [ ]:
def most_active_class(timestamp_ms):

    sender_message_count = {}
    name_class = {}

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            name = message['sender_name']
            sender_message_count[name] = sender_message_count.get(name, 0) + 1

    for i in names:
        if i['name'] in sender_message_count:
            name_class[i['name']] = i['class']

    class_message_count = {}
    class_members_count = {}

    for name, msg_count in sender_message_count.items():

        if name in name_class:
            cls = name_class[name]

            class_message_count[cls] = class_message_count.get(cls, 0) + msg_count
            class_members_count[cls] = class_members_count.get(cls, 0) + 1

    class_avg = {
        cls: class_message_count[cls] / class_members_count[cls]
        for cls in class_message_count
    }

    top5 = sorted(class_avg.items(), key=lambda x: (-x[1], x[0]))[:5]

    return top5

tuples = most_active_class(1596988217)

for t in tuples:
    print(t[0])

1
42
38
52
48


23. Megadok egy timestamp-et, külön-külön hány fotót/gifet/videót töltöttek fel addig a jeszk momentsbe? (2 pont)

In [47]:
def count_photos_videos_gifs(timestamp_ms):

    photo_count = 0
    video_count = 0
    gif_count = 0

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            photo_count += message.get('photos', 0)
            video_count += message.get('videos', 0)
            gif_count += message.get('gifs', 0)

    return photo_count, gif_count,video_count

p,g,v = count_photos_videos_gifs(1726853539100)
print(p)
print(g)
print(v)

5950
405
534


24.  Megadok egy timestamp-et, ki az eddig az időpontig írt leghosszabb üzenet szerzője? (2 pont)

In [46]:
def longest_message_sender(timestamp_ms):

    messages = []

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            name = message['sender_name']
            length = message['content_l']

            messages.append((name, length))

    # top 5 messages (length DESC, name ASC tie-break)
    top5 = sorted(
        messages,
        key=lambda x: (-x[1], x[0])
    )[:5]

    return top5

tuples = longest_message_sender(1726853539100)

for t in tuples:
    print(t[0])

Barnabás Muzslai
Marcell Bíró
Barnabás Felicián Tóth
Barnabás Felicián Tóth
Barnabás Felicián Tóth


25.  Megadok egy timestamp-et, melyik óra eddig az időpontig a legkevésbé aktív, amiben legalább egy interakció lezajlott? (tehát nem 0 az aktivitás) (2 pont)

In [45]:
def least_active_hours(timestamp_ms):

    hour_message_count = {}

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            hour = message['hour']
            hour_message_count[hour] = hour_message_count.get(hour, 0) + 1

    # bottom 5 (count ASC, hour ASC tie-break)
    bottom5 = sorted(
        hour_message_count.items(),
        key=lambda x: (x[1], x[0])
    )[:5]

    return bottom5

tuples = least_active_hours(1726853539100)

for t in tuples:
    print(t[0])

6
5
7
4
3


26.  Megadok egy timestamp-et és egy embert, mondd meg, hány karaktert küldött eddig az időpontig (2 pont)

In [ ]:
def characters_so_far(timestamp_ms, person_name):

    characters = 0

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            if message['sender_name'] == person_name:
                characters += message['content_l']

    return characters

chars = characters_so_far(1726853539100, "Zsuzsi Szentesi")
print(chars)

30562


27.  Megadok egy timestamp-et, összesen hány különböző ember írt, összesen hány üzenetet és hány karaktert írtak eddig az időpontig? (4 pont)
    - +2 pontért: emberenként csoportosítva hány üzenetet és hány karaktert küldtek (4 pont)

In [43]:
def num_senders_messages_characters(timestamp_ms):

    senders = set()
    num_messages = 0
    num_characters = 0

    for message in merged:
        if message['timestamp_ms'] <= timestamp_ms:

            num_messages += 1
            num_characters += message['content_l']

            senders.add(message['sender_name'])

    return (
        len(senders),
        num_messages,
        num_characters
    )

p,m,c = num_senders_messages_characters(1726853539100)

print(p)
print(m)
print(c)

252
55132
1964123


28.  Megadok egy órát, átlagosan hány üzenetet küldtek ebben az órában az összes olyan napra átlagolva, amikor küldtek üzenetet abban az órában? (5 pont)

In [71]:
def nepszeru_orak(ora):

    day_counts = {}

    for i in merged:

        if i['hour'] == ora:

            key = (i['year'], i['month'], i['day'])

            day_counts[key] = day_counts.get(key, 0) + 1

    if not day_counts:
        return 0

    return sum(day_counts.values()) / len(day_counts)

print(nepszeru_orak(4))

2.371212121212121


29.  Megadok egy timestamp-et, mi volt a leghosszabb periódus eddig az időpontig üzenet nélkül (5 pont)

In [74]:
def leghosszabb_szunet(timestamp):

    timestamps = [
        i['timestamp_ms']
        for i in merged
        if i['timestamp_ms'] < timestamp
    ]

    timestamps.sort()

    szunetek = [
        timestamps[i+1] - timestamps[i]
        for i in range(len(timestamps) - 1)
    ]

    szunetek.sort(reverse=True)

    return szunetek[:5]

lista = leghosszabb_szunet(1726853539100)

for l in lista:
    print(l)

733456577
485197076
405675042
370900396
350575271


30. Megadok egy timestampet, ki kapta eddig az időpontig összesen a legtöbb reakciót? (2 pont)

In [76]:
def legtobb_reakcio(timestamp):

    reaction_count = {}

    for i in merged:
        if i['timestamp_ms'] < timestamp:

            name = i['sender_name']
            reaction_count[name] = reaction_count.get(name, 0) + len(i.get('reactions', []))

    top5 = sorted(
        reaction_count.items(),
        key=lambda x: (-x[1], x[0])
    )[:5]

    return top5

tuples = legtobb_reakcio(1726853539100)

for t in tuples:
    print(t[0])

Attila Chikan
Ákos Ozoróczy
Zoli Laufer
Veronika Ipacs
Márk Mendöl


31. Az 57. évfolyamról ki küldte a legtöbb eseményt? (1 pont)

In [25]:
from collections import Counter

# 57. évfolyamosok nevei
evfolyam = {n['name'] for n in names if n['class'] == 57}

# Események megszámlálása
esemenyek = Counter(
    i['sender_name']
    for i in merged
    if i.get('sender_name') in evfolyam
)

# Rendezés csökkenő sorrendben
rendezett = esemenyek.most_common()
# Az 5. helyezett eseményszáma
if len(rendezett) >= 5:
    hatar = rendezett[4][1]
else:
    hatar = 0

# Top 5, holtversenyekkel
for nev, db in rendezett:
    if db >= hatar:
        print(nev)

Ádám Kaló
Réka Foky
József Gajzágó
Oliver Vida
Izabell Dross


32. Az 57. évfolyam eseményeire ki reagált a legtöbbet? (2 pont)

In [27]:
from collections import Counter

# 57. évfolyamosok nevei
evfolyam = {n['name'] for n in names if n['class'] == 57}

# Reakciók összeszámolása
reakciok = Counter()

for i in merged:
    if i.get('sender_name') in evfolyam:
        for r in i.get('reactions', []):
            reakciok[r['actor']] += 1

# Rendezés csökkenő sorrendben
rendezett = reakciok.most_common()

# Top 5, holtversenyek figyelembevételével
if len(rendezett) >= 5:
    hatar = rendezett[4][1]
else:
    hatar = 0

for nev, db in rendezett:
    if db >= hatar:
        print(nev)

Kincső Boróka Pénzes
Zsigmond Goreczky
Marcell Pap
Péter Aczél
Máté Fischer


33. Ki reagál az 57. évfolyamból a legtöbbet a jeszk moments eseményekre? (1 pont)

In [30]:
from collections import Counter

# 57. évfolyamosok nevei
evfolyam = {n['name'] for n in names if n['class'] == 57}

# Reakciók összeszámolása az összes eseményen
reakciok = Counter()

for i in merged:
    for r in i.get('reactions', []):
        if r['actor'] in evfolyam:
            reakciok[r['actor']] += 1

# Rendezés csökkenő sorrendben
rendezett = reakciok.most_common()

# Top 5, holtversenyekkel
if len(rendezett) >= 5:
    hatar = rendezett[4][1]
else:
    hatar = 0

for nev, db in rendezett:
    if db >= hatar:
        print(nev)

Marcell Pap
Zabolai Dorina
Oliver Vida
Viktória Keresztényi
Ádám Kaló


34. Igaz-e, hogy Attila több üzenetet küldött, mint a mostani igazgatók összesen? (2 pont)

In [33]:
from collections import Counter

# Üzenetek számlálása személyenként
uzenetek = Counter(
    i['sender_name']
    for i in merged
    if 'sender_name' in i
)

attila = uzenetek["Attila Chikan"]

tobbiek = (
    uzenetek["Anita Kaderják"]
    + uzenetek["Daniel Horn"]
    + uzenetek["Marton Kovesdy"]
)

if attila > tobbiek:
    print("Igaz")
else:
    print("Hamis")

Igaz


35. Ki az aki a legaktívabb (legtöbb eseményt küldi) 20:00 és 20:15 között (inkluzív). (3 pont)

In [36]:
from collections import Counter

esemenyek = Counter(
    i['sender_name']
    for i in merged
    if i['hour'] == 20 and 0 <= i['minute'] <= 15
)

rendezett = esemenyek.most_common()

if rendezett:
    hatar = rendezett[min(4, len(rendezett) - 1)][1]

    for nev, db in rendezett:
        if db >= hatar:
            print(nev)
else:
    print("Nincs esemény ebben az időintervallumban.")

Ákos Ozoróczy
Hobot Peter
Anna Laczkovich
Eszter Kaló
Kamilla Pelle


36. A prog kurzustagok 2026-ban a kurzustagok hozzászólásainak összegéhez mekkora arányában járultak hozzá? (4 pont)

In [75]:
from collections import Counter

nevek = [
    "Balázs Gáti",
    "Drong Lilla",
    "József Gajzágó",
    "Viktória Keresztényi"
]

for nev in nevek:
    uzenetek = [
        i for i in merged
        if i['sender_name'] == nev and len(i.get('content', '')) > 0
    ]

    osszes = len(uzenetek)

    uzenetek_2026 = sum(
        1 for i in uzenetek if i['year'] == 2026
    )

    if osszes > 0:
        arany = uzenetek_2026 / osszes * 100
        print(f"{arany:.2f}")
    else:
        print(f"{nev}: nincs üzenet")

23.86
100.00
73.28
53.12
